# EE 451: Communications Systems
## Lecture 20 - M-ary Modulation, QAM & EVM

### Learning Objectives

By the end of this notebook, you will be able to:

1. Extend from binary to M-ary signaling for increased spectral efficiency
2. Analyze QPSK and higher-order PSK constellations
3. Design and evaluate QAM (Quadrature Amplitude Modulation) systems
4. Calculate Error Vector Magnitude (EVM) as a modulation quality metric
5. Compare spectral efficiency across modulation schemes

### Textbook Reference
Haykin & Moher, Chapter 7.5-7.7

---

## Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq, fftshift
from scipy.special import erfc

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

## Part 1: From Binary to M-ary Signaling

Binary modulation (BPSK, FSK, ASK) transmits **1 bit per symbol**.

M-ary modulation uses $M$ distinct symbols to transmit $\log_2(M)$ **bits per symbol**:

| M | Bits/Symbol | Example |
|---|-------------|--------|
| 2 | 1 | BPSK |
| 4 | 2 | QPSK |
| 8 | 3 | 8-PSK |
| 16 | 4 | 16-QAM |
| 64 | 6 | 64-QAM |
| 256 | 8 | 256-QAM |

**Key tradeoff:** Higher $M$ gives more bits/symbol (spectral efficiency), but symbols are closer together, requiring higher SNR.

In [ ]:
# M-ary spectral efficiency scaling
M_values = [2, 4, 8, 16, 64, 256]
bits_per_symbol = [np.log2(M) for M in M_values]
labels = ['BPSK', 'QPSK', '8-PSK', '16-QAM', '64-QAM', '256-QAM']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: bits per symbol
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(M_values)))
bars = axes[0].bar(labels, bits_per_symbol, color=colors, edgecolor='black')
for bar, bps in zip(bars, bits_per_symbol):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{int(bps)}', ha='center', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Bits per Symbol', fontsize=12)
axes[0].set_title('M-ary Modulation: Bits per Symbol', fontsize=14)
axes[0].set_ylim(0, 10)

# Data rate advantage at fixed symbol rate
R_s = 1e6  # 1 Msym/s
data_rates = [R_s * bps for bps in bits_per_symbol]
axes[1].bar(labels, [r/1e6 for r in data_rates], color=colors, edgecolor='black')
axes[1].set_ylabel('Data Rate (Mbps)', fontsize=12)
axes[1].set_title(f'Data Rate at {R_s/1e6:.0f} Msym/s Symbol Rate', fontsize=14)

plt.tight_layout()
plt.show()

print("M-ary Signaling Summary:")
print("═" * 60)
print(f"{'Modulation':<12} {'M':<6} {'Bits/Sym':<12} {'Rate at 1 Msym/s':<18}")
print("─" * 60)
for label, M, bps, rate in zip(labels, M_values, bits_per_symbol, data_rates):
    print(f"{label:<12} {M:<6} {int(bps):<12} {rate/1e6:.0f} Mbps")
print("═" * 60)
print("\nSame symbol rate → Same bandwidth → Higher data rate with larger M")

## Part 2: QPSK (Quadrature Phase Shift Keying)

QPSK uses $M = 4$ phase states to transmit **2 bits per symbol**:

$$s(t) = A \cos(2\pi f_c t + \theta_k), \quad \theta_k \in \{45°, 135°, 225°, 315°\}$$

With Gray coding:
- 00 → 45° → $(+1, +1)/\sqrt{2}$
- 01 → 135° → $(-1, +1)/\sqrt{2}$  
- 11 → 225° → $(-1, -1)/\sqrt{2}$
- 10 → 315° → $(+1, -1)/\sqrt{2}$

QPSK can be viewed as **two independent BPSK signals** on I and Q channels.

In [ ]:
# QPSK constellation with Gray coding
qpsk_map = {
    '00': (1 + 1j) / np.sqrt(2),
    '01': (-1 + 1j) / np.sqrt(2),
    '11': (-1 - 1j) / np.sqrt(2),
    '10': (1 - 1j) / np.sqrt(2)
}

# QPSK signal generation
R_b = 2000      # Bit rate (bps)
R_s = R_b / 2   # Symbol rate (1000 sym/s for QPSK)
f_c = 10000     # Carrier frequency
fs = 100000     # Sampling rate
num_bits = 20

np.random.seed(42)
bits = np.random.randint(0, 2, num_bits)

# Map bit pairs to symbols
symbols = []
for i in range(0, len(bits), 2):
    key = f'{bits[i]}{bits[i+1]}'
    symbols.append(qpsk_map[key])
symbols = np.array(symbols)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Constellation diagram
for label, point in qpsk_map.items():
    axes[0].plot(point.real, point.imag, 'ro', markersize=20)
    axes[0].annotate(label, (point.real, point.imag),
                     textcoords='offset points', xytext=(12, 12),
                     fontsize=14, fontweight='bold')

# Draw unit circle and decision boundaries
theta = np.linspace(0, 2*np.pi, 100)
axes[0].plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3)
axes[0].axhline(y=0, color='green', linestyle='--', linewidth=2, alpha=0.5, label='Decision boundary')
axes[0].axvline(x=0, color='green', linestyle='--', linewidth=2, alpha=0.5)
axes[0].set_xlabel('In-Phase (I)', fontsize=12)
axes[0].set_ylabel('Quadrature (Q)', fontsize=12)
axes[0].set_title('QPSK Constellation (Gray Coded)', fontsize=14)
axes[0].set_xlim(-1.5, 1.5)
axes[0].set_ylim(-1.5, 1.5)
axes[0].set_aspect('equal')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Show I and Q as independent BPSK
T_s = 1 / R_s
t_sym = np.arange(0, len(symbols) * T_s, 1/fs)
I_signal = np.zeros(len(t_sym))
Q_signal = np.zeros(len(t_sym))
for n, sym in enumerate(symbols):
    start = int(n * T_s * fs)
    end = int((n + 1) * T_s * fs)
    if end <= len(I_signal):
        I_signal[start:end] = sym.real
        Q_signal[start:end] = sym.imag

t_plot = t_sym[:int(5 * T_s * fs)] * 1000  # First 5 symbols
axes[1].plot(t_plot, I_signal[:len(t_plot)], linewidth=2, label='I channel (even bits)')
axes[1].plot(t_plot, Q_signal[:len(t_plot)], linewidth=2, label='Q channel (odd bits)')
axes[1].set_xlabel('Time (ms)', fontsize=12)
axes[1].set_ylabel('Amplitude', fontsize=12)
axes[1].set_title('QPSK I and Q Channels (Two Independent BPSK)', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].set_ylim(-1.2, 1.2)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"QPSK Parameters:")
print(f"  Bit rate: R_b = {R_b} bps")
print(f"  Symbol rate: R_s = R_b/2 = {R_s:.0f} sym/s")
print(f"  Bits per symbol: 2")
print(f"  Bandwidth: ~2R_s = {2*R_s:.0f} Hz (same as BPSK at {R_s:.0f} sym/s)")
print(f"  Spectral efficiency: 1 bit/s/Hz (double BPSK's 0.5)")

## Part 3: Higher-Order PSK and QAM Constellations

### M-PSK: Points on a Circle
- Phase states equally spaced around unit circle
- **Problem:** As $M$ increases, points get closer → more noise-sensitive

### QAM: Points on a Grid
- Vary **both** amplitude and phase
- Rectangular grid is more efficient than circular PSK for $M \geq 16$
- 16-QAM: $4 \times 4$ grid, 4 bits/symbol
- 64-QAM: $8 \times 8$ grid, 6 bits/symbol
- 256-QAM: $16 \times 16$ grid, 8 bits/symbol

In [ ]:
def generate_mpsk(M):
    """Generate M-PSK constellation points."""
    angles = np.array([2 * np.pi * k / M + np.pi/M for k in range(M)])
    return np.exp(1j * angles)

def generate_mqam(M):
    """Generate square M-QAM constellation points."""
    sqrt_M = int(np.sqrt(M))
    levels = np.arange(-(sqrt_M - 1), sqrt_M, 2)
    points = []
    for i in levels:
        for q in levels:
            points.append(i + 1j * q)
    points = np.array(points)
    # Normalize to unit average power
    return points / np.sqrt(np.mean(np.abs(points)**2))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Row 1: M-PSK constellations
for idx, (M, ax) in enumerate(zip([4, 8, 16], axes[0])):
    points = generate_mpsk(M)
    ax.plot(points.real, points.imag, 'ro', markersize=12)
    theta = np.linspace(0, 2*np.pi, 100)
    ax.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.set_xlim(-1.8, 1.8)
    ax.set_ylim(-1.8, 1.8)
    ax.set_aspect('equal')
    ax.set_title(f'{M}-PSK ({int(np.log2(M))} bits/sym)', fontsize=13)
    ax.grid(True, alpha=0.3)
    # Show minimum distance
    d_min = np.min([np.abs(points[i] - points[j])
                    for i in range(len(points)) for j in range(i+1, len(points))])
    ax.text(0.05, 0.95, f'd_min = {d_min:.3f}', transform=ax.transAxes,
            fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Row 2: QAM constellations
for idx, (M, ax) in enumerate(zip([16, 64, 256], axes[1])):
    points = generate_mqam(M)
    ax.plot(points.real, points.imag, 'bs', markersize=8 if M <= 64 else 4)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    lim = np.max(np.abs(points)) * 1.3
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.set_title(f'{M}-QAM ({int(np.log2(M))} bits/sym)', fontsize=13)
    ax.grid(True, alpha=0.3)
    d_min = np.min([np.abs(points[i] - points[j])
                    for i in range(len(points)) for j in range(i+1, len(points))])
    ax.text(0.05, 0.95, f'd_min = {d_min:.3f}', transform=ax.transAxes,
            fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

axes[0, 0].set_ylabel('Quadrature (Q)', fontsize=12)
axes[1, 0].set_ylabel('Quadrature (Q)', fontsize=12)
for ax in axes[1]:
    ax.set_xlabel('In-Phase (I)', fontsize=12)

plt.suptitle('M-PSK (top) vs M-QAM (bottom): Constellation Comparison', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

print("Key Observation:")
print("  PSK: Points on a circle → d_min shrinks rapidly with M")
print("  QAM: Points on a grid  → d_min shrinks more slowly")
print("  For M >= 16, QAM is more power-efficient than PSK")

## Part 4: Spectral Efficiency Comparison

**Spectral efficiency** $\eta$ measures how efficiently bandwidth is used:

$$\eta = \frac{R_b}{B} = \frac{\log_2(M) \cdot R_s}{B} \quad \text{bits/s/Hz}$$

For Nyquist pulse shaping with roll-off $\alpha$: $B = (1+\alpha) R_s$

With $\alpha = 0$ (ideal): $\eta = \log_2(M)$ bits/s/Hz

### WiFi and LTE Adaptive Modulation
Modern systems select modulation based on channel quality:
- Good SNR → 256-QAM (high rate)
- Poor SNR → QPSK or BPSK (robust)
- **MCS (Modulation and Coding Scheme)** index controls this adaptation

In [ ]:
# Spectral efficiency comparison
modulations = ['BPSK', 'QPSK', '8-PSK', '16-QAM', '64-QAM', '256-QAM', '1024-QAM']
M_vals = [2, 4, 8, 16, 64, 256, 1024]
spec_eff = [np.log2(M) for M in M_vals]

# Approximate required Eb/N0 for BER = 1e-5 (QAM values from standard tables)
required_snr_db = [9.6, 9.6, 14.0, 13.4, 17.8, 24.0, 28.5]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart of spectral efficiency
colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(modulations)))
bars = axes[0].bar(modulations, spec_eff, color=colors, edgecolor='black')
for bar, eff in zip(bars, spec_eff):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
                 f'{eff:.1f}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Spectral Efficiency (bits/s/Hz)', fontsize=12)
axes[0].set_title('Spectral Efficiency by Modulation', fontsize=14)
axes[0].tick_params(axis='x', rotation=30)

# Spectral efficiency vs required SNR scatter
for i, (mod, eff, snr) in enumerate(zip(modulations, spec_eff, required_snr_db)):
    axes[1].plot(snr, eff, 'o', markersize=15, color=colors[i])
    axes[1].annotate(mod, (snr, eff), textcoords='offset points',
                     xytext=(8, 5), fontsize=10)

# Shannon limit
snr_range = np.linspace(0, 32, 200)
shannon_limit = np.log2(1 + 10**(snr_range/10))
axes[1].plot(snr_range, shannon_limit, 'k--', linewidth=2, alpha=0.5, label='Shannon Limit')
axes[1].fill_between(snr_range, 0, shannon_limit, alpha=0.05, color='gray')

axes[1].set_xlabel('Required SNR per bit (dB) for BER = 10⁻⁵', fontsize=12)
axes[1].set_ylabel('Spectral Efficiency (bits/s/Hz)', fontsize=12)
axes[1].set_title('Spectral Efficiency vs Required SNR', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].set_xlim(5, 32)
axes[1].set_ylim(0, 12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nSpectral Efficiency Summary:")
print("═" * 65)
print(f"{'Modulation':<12} {'Bits/Sym':<10} {'η (bits/s/Hz)':<16} {'Req. SNR (dB)':<14}")
print("─" * 65)
for mod, M, eff, snr in zip(modulations, M_vals, spec_eff, required_snr_db):
    print(f"{mod:<12} {int(eff):<10} {eff:<16.1f} {snr:<14.1f}")
print("═" * 65)
print("\nWiFi 6 (802.11ax): Uses up to 1024-QAM (10 bits/symbol)")
print("LTE: Uses up to 256-QAM (8 bits/symbol)")
print("5G NR: Uses up to 256-QAM (downlink), 64-QAM (uplink)")

## Part 5: Error Vector Magnitude (EVM)

EVM measures how far received symbols deviate from their ideal positions:

$$\text{EVM} = \sqrt{\frac{\frac{1}{N}\sum_{n=1}^{N}|\mathbf{r}_n - \mathbf{s}_n|^2}{P_{\text{avg}}}} \times 100\%$$

where $\mathbf{r}_n$ is the received symbol and $\mathbf{s}_n$ is the ideal symbol.

### EVM Specifications
| Standard | Modulation | Max EVM |
|----------|------------|--------|
| WiFi 802.11ax | 256-QAM | −30 dB (3.16%) |
| LTE | 64-QAM | −22 dB (8%) |
| 5G NR | 256-QAM | −27 dB (4.5%) |

In [ ]:
# Worked Example: EVM calculation (from lesson plan)
print("Worked Example: EVM for a Single QPSK Symbol")
print("═" * 55)

ideal = np.array([1.0, 0.0])       # Ideal symbol on I-Q plane
received = np.array([0.95, 0.10])   # Received with noise/distortion

error = received - ideal
error_mag = np.sqrt(error[0]**2 + error[1]**2)
ref_mag = np.sqrt(ideal[0]**2 + ideal[1]**2)
evm_single = (error_mag / ref_mag) * 100

print(f"Ideal symbol:    ({ideal[0]:.2f}, {ideal[1]:.2f})")
print(f"Received symbol: ({received[0]:.2f}, {received[1]:.2f})")
print(f"Error vector:    ({error[0]:.2f}, {error[1]:.2f})")
print(f"Error magnitude: √({error[0]:.2f}² + {error[1]:.2f}²) = {error_mag:.4f}")
print(f"Reference:       {ref_mag:.2f}")
print(f"EVM = {error_mag:.4f} / {ref_mag:.2f} = {evm_single:.1f}%")
print(f"EVM in dB = 20·log₁₀({evm_single/100:.4f}) = {20*np.log10(evm_single/100):.1f} dB")

# Visualize
fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(ideal[0], ideal[1], 'go', markersize=18, label='Ideal', zorder=5)
ax.plot(received[0], received[1], 'r^', markersize=15, label='Received', zorder=5)
ax.annotate('', xy=received, xytext=ideal,
            arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
ax.text(0.975, 0.06, f'Error vector\n|e| = {error_mag:.3f}', fontsize=11,
        color='red', ha='center')

# Show other QPSK points
for label, pt in qpsk_map.items():
    ax.plot(pt.real, pt.imag, 'ko', markersize=10, alpha=0.3)

theta_c = np.linspace(0, 2*np.pi, 100)
ax.plot(np.cos(theta_c)/np.sqrt(2), np.sin(theta_c)/np.sqrt(2), 'k--', alpha=0.2)
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.set_xlabel('In-Phase (I)', fontsize=12)
ax.set_ylabel('Quadrature (Q)', fontsize=12)
ax.set_title(f'EVM = {evm_single:.1f}% ({20*np.log10(evm_single/100):.1f} dB)', fontsize=14)
ax.set_xlim(-0.3, 1.3)
ax.set_ylim(-0.3, 0.5)
ax.set_aspect('equal')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# EVM vs SNR for different modulations
def calculate_evm(transmitted, received):
    """Calculate EVM in percent."""
    error_power = np.mean(np.abs(received - transmitted)**2)
    signal_power = np.mean(np.abs(transmitted)**2)
    return np.sqrt(error_power / signal_power) * 100

def add_awgn(symbols, snr_db):
    """Add AWGN to complex symbols."""
    sig_power = np.mean(np.abs(symbols)**2)
    noise_power = sig_power / (10**(snr_db/10))
    noise = np.sqrt(noise_power/2) * (np.random.randn(len(symbols)) +
                                       1j * np.random.randn(len(symbols)))
    return symbols + noise

# Generate symbols for each modulation
np.random.seed(0)
N = 5000
qpsk_pts = generate_mpsk(4)
qam16_pts = generate_mqam(16)
qam64_pts = generate_mqam(64)

qpsk_syms = qpsk_pts[np.random.randint(0, 4, N)]
qam16_syms = qam16_pts[np.random.randint(0, 16, N)]
qam64_syms = qam64_pts[np.random.randint(0, 64, N)]

snr_range = np.arange(5, 35, 1)
evm_qpsk = [calculate_evm(qpsk_syms, add_awgn(qpsk_syms, s)) for s in snr_range]
evm_16qam = [calculate_evm(qam16_syms, add_awgn(qam16_syms, s)) for s in snr_range]
evm_64qam = [calculate_evm(qam64_syms, add_awgn(qam64_syms, s)) for s in snr_range]

# Theoretical: EVM = 100 / sqrt(SNR_linear)
evm_theory = 100 / np.sqrt(10**(snr_range/10))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# EVM vs SNR
axes[0].semilogy(snr_range, evm_qpsk, 'b-o', markersize=4, label='QPSK')
axes[0].semilogy(snr_range, evm_16qam, 'r-s', markersize=4, label='16-QAM')
axes[0].semilogy(snr_range, evm_64qam, 'g-^', markersize=4, label='64-QAM')
axes[0].semilogy(snr_range, evm_theory, 'k--', linewidth=2, label='Theoretical')

# WiFi/LTE/5G spec lines
axes[0].axhline(y=3.16, color='purple', linestyle=':', alpha=0.7, label='WiFi 256-QAM (3.16%)')
axes[0].axhline(y=8.0, color='orange', linestyle=':', alpha=0.7, label='LTE 64-QAM (8%)')
axes[0].axhline(y=4.5, color='brown', linestyle=':', alpha=0.7, label='5G NR 256-QAM (4.5%)')

axes[0].set_xlabel('SNR (dB)', fontsize=12)
axes[0].set_ylabel('EVM (%)', fontsize=12)
axes[0].set_title('EVM vs SNR', fontsize=14)
axes[0].legend(fontsize=9, loc='upper right')
axes[0].set_ylim(1, 100)
axes[0].grid(True, alpha=0.3, which='both')

# Noisy constellations at SNR = 15 dB
snr_demo = 15
noisy_16qam = add_awgn(qam16_syms[:500], snr_demo)
axes[1].scatter(noisy_16qam.real, noisy_16qam.imag, alpha=0.3, s=8, c='blue')
axes[1].plot(qam16_pts.real, qam16_pts.imag, 'r*', markersize=15, zorder=5)
axes[1].set_xlabel('In-Phase (I)', fontsize=12)
axes[1].set_ylabel('Quadrature (Q)', fontsize=12)
evm_demo = calculate_evm(qam16_syms[:500], noisy_16qam)
axes[1].set_title(f'16-QAM at SNR = {snr_demo} dB (EVM = {evm_demo:.1f}%)', fontsize=14)
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nEVM-SNR Relationship:")
print("  For AWGN: EVM (%) ≈ 100 / √(SNR_linear)")
print("  EVM (dB) = −SNR (dB) / 2")
print(f"\n  Example: SNR = 20 dB → EVM = 100/√100 = 10%")
print(f"  Example: SNR = 30 dB → EVM = 100/√1000 = 3.16%")

## Summary and Key Takeaways

### M-ary Modulation
- $M$ symbols → $\log_2(M)$ bits per symbol
- Same symbol rate, same bandwidth → higher data rate
- **Tradeoff:** Higher $M$ requires better SNR

### QPSK
- 4 phase states, 2 bits/symbol, Gray coding
- Two independent BPSK channels (I and Q)
- Same BER as BPSK at same $E_b/N_0$
- Spectral efficiency: 1 bit/s/Hz

### QAM
- Vary both amplitude and phase
- Rectangular grid more efficient than M-PSK for $M \geq 16$
- WiFi: up to 1024-QAM, LTE: up to 256-QAM
- Adaptive modulation selects MCS based on channel quality

### EVM
- Measures constellation accuracy (RMS error / reference)
- For AWGN: $\text{EVM} \approx 100/\sqrt{\text{SNR}}$
- WiFi 256-QAM requires EVM < 3.16% (−30 dB)
- LTE 64-QAM requires EVM < 8% (−22 dB)

### Spectral Efficiency
| Modulation | Bits/Symbol | η (bits/s/Hz) |
|------------|------------|----------------|
| BPSK | 1 | 0.5 |
| QPSK | 2 | 1 |
| 16-QAM | 4 | 2 |
| 64-QAM | 6 | 3 |
| 256-QAM | 8 | 4 |

### Next Topics
- FT8: 8-FSK weak-signal digital mode
- Costas arrays for synchronization
- Midterm Exam 2 (Lesson 21)